# RSX-E08 — Repair locality and decoder ablations (GPU)

**72 new repair trials on six distinct NB29H calibration failures.** All experiments are exploratory. The study directly tests whether preserving text before the first visible arithmetic operation is more or less effective than fixed-tail repair, answer-locus repair, or full regeneration, using left1 and entropy decoding.

Case selection is frozen, includes **no held-out-labeled problems**, and is disjoint from E09. No original generation or completed P1 trial is repeated. New interventions are saved immediately, with a session limit and recoverable Drive files.

**Important:** `first_math` is a surface-text heuristic, **not** a validated first-error locator. Normalized efficiency and matched-budget comparisons are separate claims.

In [ ]:
import os,sys,subprocess
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
REPO_DIR='/content/ALoDLM_RSX_E08E09'
if not os.path.isdir(os.path.join(REPO_DIR,'.git')):
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',REPO_DIR],check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1.0','safetensors>=0.6','PyYAML>=6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO_DIR,'--no-deps'],check=True)
print('Pinned source:',CODE_COMMIT)

In [ ]:
import json,time,random,hashlib,math,zipfile,gc,os,re,ast,base64,zlib
from pathlib import Path
from types import SimpleNamespace
from collections import Counter
import torch, numpy as np, pandas as pd
from huggingface_hub import snapshot_download
from google.colab import drive,files
from IPython.display import display
if not torch.cuda.is_available():raise RuntimeError('Colab GPU required (T4 or better).')
if torch.cuda.get_device_properties(0).total_memory<12*2**30:raise RuntimeError('Need >=12 GiB GPU.')
drive.mount('/content/drive')
MODEL_ID='amazon/ALoDLM-1.7B';MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
Q=0.4;TAU=0.4;WINDOW=16
PROMPTS={'concise':'\n\nGive a short solution. End with one line: FINAL_ANSWER: <number>. For Countdown include an expression using all inputs once.',
'answer_first':'\n\nStart with FINAL_ANSWER: <number> on the first line, then optionally explain. For Countdown show the expression using each number exactly once.',
'structured':'\n\nUse exactly this format: ANSWER: <number>\nCHECK: <short verification>. For Countdown also show an expression using all input numbers exactly once.'}
def digest(v):return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
def atomic_json(path,obj):
    tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(obj,sort_keys=True,indent=2,ensure_ascii=False));os.replace(tmp,path)
def verify_save(rec):
    dest=RECS/(hashlib.sha256(rec['trial_id'].encode()).hexdigest()+'.json')
    if dest.exists():
        prior=json.loads(dest.read_text());assert prior==rec,'Conflict: refusing overwrite '+rec['trial_id']
    else:atomic_json(dest,rec)
def checked_load(tid):
    dest=RECS/(hashlib.sha256(tid.encode()).hexdigest()+'.json')
    if not dest.exists():return None
    r=json.loads(dest.read_text());assert r['trial_id']==tid and r['protocol_hash']==PROTOCOL_HASH
    return r
def recorded():
    v=[json.loads(p.read_text()) for p in sorted(RECS.glob('*.json'))]
    assert len(v)==len({r['trial_id'] for r in v})
    assert all(r['protocol_hash']==PROTOCOL_HASH for r in v)
    return v
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
def render(ids):return tokenizer.decode(list(ids),skip_special_tokens=True).strip()
def make_prompt(content):return list(tokenizer.apply_chat_template([{'role':'user','content':content}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
def snapshot_export(name):
    out=Path('/content')/name
    with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(ROOT.rglob('*')):
            if p.is_file() and p.suffix!='.tmp':z.write(p,p.relative_to(ROOT))
    print('Exported',out,'records:',len(recorded()))
    files.download(str(out))
    return out

STUDY='E08_RepairLocality_calibration_v1'
ROOT=Path('/content/drive/MyDrive/RepairScopeX/E08_RepairLocality_V1');RECS=ROOT/'records';RECS.mkdir(parents=True,exist_ok=True)
DATA=json.loads(zlib.decompress(base64.b64decode("eNrtWmtv3LgV/SuE+qHZRGuIb9LoA2mbxS5Q5EO2L2AdTCiJ8giZkWb1iJMm/u89lDQzGr8ydtI0iy38gEckLw8vL8+5l/L7KHOtb6PTn95Hm6ZOV34dnb6Psnq1KtuyrhaN75p30WkSR6/LKo9OI9eU3XLtuzKL4qjq16lvwvCX8Xb8oiirc99smrLq0J9niZcqY7owmrNcp15LS61mRc4ELTRTMk+5oirPeOELp1PhWUGLTHibOaWiveEyzN+0b5k9zdyqTBvXAeLpHtFpkgg6DlhvwuRPSbv0q4Is61XeEs1IWtev2xPy7K3LutU7wsz4hLjG46+mqS98fkK+ry/I2lXvpsbGr11Z/ZE8q3JygbnIdz88f/rXxdPnP/7z2YtT8rvRC384wcztZlWGiWf48LRzzbnHYyFO4MjOrzcr1/nolF5iRN03mQ9OT/t86GVVHPk3btWPw9HiqvbCN4u6Kc/LKuzB+HmzbLB5sO9XaICLFi2QVnDEonDlqm9gtmt6jw5vgSsru7A3brUYAQNPgDN1XXTvNugf5R59ys4vLpq6Oh/32DcwmcE9PgO+wq1a2Gw7PO2uP+6zzLft7jMWUuYLAGjwdFhP1a9WcTSYv/YcDjn3lYfnfL7o6te+akeHnDd1vwE8hFXdLNw6Lc/7um+jm+NyXedhLYDYZx0Wl+NZ3XebPuzN35Z+Couw0NKtEAhLNw+Pp0XnmykeEMwIk5h0GDUGQniyjxrsdNaH7cwJbBQ4OPVFe3pW4evsp7MKRr8NYfZ7IkR49DK0PH78D7i0KLMxgB8/JuSseprnwXKYZxuJ0zRdfWX28/KNb8nZ2SMhyJPRumZnZ9/EJKuromzWW0NbcJiFlC2Z9uokTBdQPNt5n/RtGANfkOlMk7rK/AQNMx2s4+yb0cL2BIgHHtKuBtEs4PslxqSZYEUicp3nBbUiTb1wSZ64wqSJSgvpnbJW58pKY3OaFpmnaabBIrnIGXUynD6PnT61lllBbzuLbVdvwGquDSEXrXx13i3DEQ2xBvCBy7SRMdNKyJhKmrBY44exBD8x1bESSseUx0JxFUsmtRwa0cJ4TClG2tgqJcaO6BBTJY2JBTdoTtCPCoGG0EtsjQpDZ1aYEeMnO36b0JtJzmPOYZdxbmOlrEJPnuAhfpthXiHldmJ+AAWdYVsH5INNAUDX5xuxYC6dUHSxGBRMMGo5DGqBNQQ7WKZRMB/AWGnDAmFYacCkiY15IuUcnhDMsiPWKbWWZjJNqcIYbrlE/8nBFhoDzgF/HRdjH57/idnvP+xJ4INVH8bgwIZfINbBlzgwYc/1SaLl5WV8rApmdV91eX1RHYhg8AxcZm4VwwJiaLxSPOOWamaSjBnDC2EzabX1mmZUFeBPBIhw1rvMc+k5YxkU1AnmPn7OdsDggiSZa+HfWz9wwvZ8AywBWuKgajjYfpLEcOwD42Bc3mfjkFG/CD0hLzw8WRFHBlone4+TPZMPFv1WK1/dLJavPq6W9FAso82qb8FsIP0Hyua6bAPLfYJgjtL11ShmRDlYmaqBmqMbtVPwI7RzHs2TdOJcZOWQXBzo5l3bPkrIkWFFoUeTSqb1W5+/ny8FKvMto5cPi/b/vaj4ur1JUZQFiQoNztPC8pE0B2IDpZpBOcYPLNBfIPUZUyqD8B6INfC6ASFvewda3gqKtmCx2G5N7sh2sHmdesOTIEcqtuxj3Hro6YlapyC5g1fFCQT0k3k1yHHMkltpVZhCSZVmqUsz76nLBLaWFQL1Q2GkxCNbuDxVqZQsdVIom1Cjc+dyIQ0G3zPQ6F20KmMCsGP4J0exKvvCtMo+N69+rnKE6a+qHomYDmyU4JeMHl6Y3ECuk8OQqbfdnGFfvXq1edcta9Dib8jH6Bb5/AwgKJPFJO07hBZa8tq31W871AwuWw4hFkyOdY1/2wE9SohgbVWPATIWOJPzSFuv+rFoqN7Uq1BsOJKXReEbX3XEbRDGsDsaDTjHBR0H6aJuXo8jUWb7N745nPpoUw87tF9rycGlBY8zpM1Dvs+QDatrkhG4fkvhbE7oVO4+hW+z6wax0BRMj/SaU01jK0QspZA74RlKGY3xbCxoqOLDxCLZFTND0q+YDWKhUBigEXoCKZLCqP3wAFgwpj4TSgsl20PjUhzA+TwT3Uv0tvXE/PDeWVGghPxk5RODVstblS+jiVAJ48pbmhiRJtrmaZpkeaaSRFpqGJTOc+ETkQiWyoQblHVJplF4uESn9zxE6i7lE1A+PimfPEb55BdWPnlV+dp+fbPmURSfv7piApv2BDsIvhU3yx3l9rMWE3uVGe64+jvjqayGxllUnJ09mkEerr8uliXkzv/cY/FEipO5xEy6Qaq6I2vXod+VULyqjkDdlrk/0MdV3fq2m4LVf+z6LCzsECWETI5Qg5iV3R7SqNQBx198u0EEkHW/6srNCurahZjt2hgdj54azRjfBr8FowdaOx15si7Plx1ZOizWEQRgTeoGA3/uy8bfqvjh4vC7EMjk6XAkdneEYx03W+zlw9jlay3gBgkMmhNKMJDooEbDjdf2DmsvQOZqCZegehPDiLH4G+/ihobZXdy8aLPbizgDMqIKNd9W0MJUfKeJQcygxiaWHOYEZ9Odndr3399XjvJOtdF2q6fUJKGaDKnGvi49vMwbasxjEO8uEuf4B8UFrB3IXQKywwbj0GzJBPzGpRxSDxtuXe9EBYEbvXqwzFmqwIUN14cYpYAh5C0Gw5EcBB9wza+nMkLTZLjXRIUtsVY4R83vMg+K7Lt9cc+aWl2pqSFBt6UWNDmhgn76baUYymp7a3LhUmkK52TBDKVOpaIwnCUq5zgIqshpkaTMpHlCqacyoxK7r2XhmVdISxAt9zv+XNx5WylmdbU9JrvQyZfNLjDfkdnFr/GmEnv2ZKzo6C3ZheRfMrm4Ek83ZxczzOEVWwLhPpnfWc47XD4s2n+hYqcO9S683DnUO3Gb3omDd09sfkk5F5AgF4FtURdfZ927bdyPeLm4x2WmsJ/OumZYJ7/9HZF3CfOMa5aiZEvwZY2Q3jOXsyzRTCUcxRvlmVWqKERg4cIYk2rmraUpvWccmjtZ18QEaMdTwo9hXfWFWVf9n3Xvej9kt8k4NV8F6x7G042sO8cM1lXXWHfe4fJh0f6LZd3pXwymD+IK68K9N7GuusKYV14Nzd7/01tZ96M27sm65j6sq8Tl5cv/zsZNNBG98BtXNj9m9cb/azFAW/x56bPXmxqsTB7Rb07+XW6iy/8ArkAcSQ==")))
CASES=DATA['cases']; CASE_BY_ID={c['problem']['problem_id']:c for c in CASES}
assert len(CASES)==6 and all(c['problem']['split']=='calibration' and c['source']['evaluation']['eligible_semantic_failure'] for c in CASES)
ACTIONS=['global','tail25','tail50','tail75','answer_locus','first_math'];DECODE_MODES=['left1','entropy']
PLAN=[{'problem_id':pid,'action':act,'decode':mode} for pid in sorted(CASE_BY_ID) for mode in DECODE_MODES for act in ACTIONS]
assert len(PLAN)==72
CONDITION={'study':STUDY,'parent_h_protocol':DATA['protocol_hash'],'code_commit':CODE_COMMIT,'model':MODEL_ID,'model_rev':MODEL_REV,
           'policies':ACTIONS,'decode_modes':DECODE_MODES,'plan':PLAN,'q':Q,'tau':TAU,'window':WINDOW,'seed':62083}
PROTOCOL_HASH=digest(CONDITION)
if (ROOT/'protocol.json').exists():assert json.loads((ROOT/'protocol.json').read_text())=={'hash':PROTOCOL_HASH,'condition':CONDITION}
else:atomic_json(ROOT/'protocol.json',{'hash':PROTOCOL_HASH,'condition':CONDITION})
if (ROOT/'frozen_plan.json').exists():assert json.loads((ROOT/'frozen_plan.json').read_text())==PLAN
else:atomic_json(ROOT/'frozen_plan.json',PLAN)
print('Ready:',len(PLAN),'saved:',len(recorded()))

In [ ]:
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
RUN_RE=re.compile(r'[0-9()+*×÷/\-+=\s]{3,}')
def closenum(a,b):return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-9,abs_tol=1e-8)
def safe_expression(s):
    try:t=ast.parse(s,mode='eval')
    except Exception:return None
    lit=[]
    def walk(n):
        if isinstance(n,ast.Expression):return walk(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (int,float):lit.append(float(n.value));return float(n.value)
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,(ast.UAdd,ast.USub)):
            a=walk(n.operand);return (-a if isinstance(n.op,ast.USub) else a)
        if isinstance(n,ast.BinOp) and isinstance(n.op,(ast.Add,ast.Sub,ast.Mult,ast.Div)):
            a,b=walk(n.left),walk(n.right)
            if isinstance(n.op,ast.Add):return a+b
            if isinstance(n.op,ast.Sub):return a-b
            if isinstance(n.op,ast.Mult):return a*b
            if abs(b)<1e-12:raise ZeroDivisionError
            return a/b
        raise ValueError('Unsupported expression')
    try:
        v=walk(t)
        if math.isfinite(v):return {'value':v,'literals':lit}
    except Exception:pass
    return None

def extract_expression(text,problem):
    cleaned=text.replace('×','*').replace('÷','/')
    for m in RUN_RE.finditer(cleaned):
        raw=m.group(0)
        # Restrict to expression strings and avoid accidentally matching a target alone.
        for part in raw.split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            result=safe_expression(ex)
            if not result:continue
            if len(result['literals'])==len(problem['numbers']) and sorted(result['literals'])==sorted(map(float,problem['numbers'])) and closenum(result['value'],problem['target']):return ex
    return None

BOX_RE=re.compile(r'\\boxed\s*\{\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}')
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)\b(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')

def last_explicit_number(text):
    """Only explicit answer markers, never final arbitrary numerals in unfinished reasoning.
    Conflicting explicit answer values => ambiguous, never an eligible failure."""
    text=str(text)
    entries=[]
    for pattern,kind in ((FINAL_RE,'final'),(BOX_RE,'boxed'),(ANSWER_RE,'answer_phrase')):
        for m in pattern.finditer(text):
            try:entries.append((m.start(),float(m.group(1)),kind))
            except (ValueError,OverflowError):pass
    if not entries:return None,'missing'
    unique=[]
    for _,v,_ in entries:
        if not any(closenum(v,w) for w in unique):unique.append(v)
    if len(unique)>1:return None,'conflicting_markers'
    entries.sort()
    return entries[-1][1],entries[-1][2]

def countdown_wrong_candidate(text,p):
    """Inspectable wrong *complete* expression consuming exactly the supplied numbers.
    Do not equate missing expressions with semantic error."""
    for m in RUN_RE.finditer(str(text).replace('×','*').replace('÷','/')):
        for part in m.group(0).split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            r=safe_expression(ex)
            if r and len(r['literals'])==len(p['numbers']) and sorted(r['literals'])==sorted(map(float,p['numbers'])):
                if not closenum(r['value'],p['target']):return ex
    return None

def grade_output(text,p):
    text=str(text)
    num,origin=last_explicit_number(text)
    sem_numeric=closenum(num,p['target'])
    last_nonempty=next((line.strip() for line in reversed(text.splitlines()) if line.strip()),'')
    fmt=FINAL_RE.fullmatch(last_nonempty)
    strict=bool(fmt and closenum(float(fmt.group(1)),p['target']))
    if p['kind']=='countdown':
        expr=extract_expression(text,p)
        success=expr is not None
        wrong_expr=None if success else countdown_wrong_candidate(text,p)
        explicit_wrong=(num is not None and not sem_numeric)
        # An explicitly wrong answer or an inspectable illegal computation, with no valid solution.
        is_wrong=bool(not success and origin!='conflicting_markers' and (explicit_wrong or wrong_expr is not None))
    else:
        expr=None;wrong_expr=None
        success=bool(sem_numeric)
        is_wrong=bool(num is not None and not sem_numeric and origin!='conflicting_markers')
    return {'success':bool(success),'numeric_correct':bool(sem_numeric),
            'strict_correct':bool(strict),'valid_expression':expr,
            'wrong_expression':wrong_expr,'answer_origin':origin,
            'explicit_final_number':num,'eligible_semantic_failure':bool(is_wrong),
            'failure_type':('definite_wrong' if is_wrong else
                   ('semantically_correct' if success else 'unanswered_or_ambiguous'))}

def public_accept(text,p):
    """Label-blind stopping gate: never consult word-problem gold targets."""
    if p['kind']=='countdown':
        # Target is explicitly present in the Countdown input; a valid expression may be verified.
        return extract_expression(str(text),p) is not None
    num,origin=last_explicit_number(str(text))
    return num is not None and origin!='conflicting_markers'

def features(text,p,n_tokens):
    """Target-free features; no test labels or correctness queries."""
    num,origin=last_explicit_number(text)
    L=int(n_tokens)
    return {'kind':p['kind'],'length_bucket':'short' if L<45 else ('medium' if L<80 else 'long'),
            'has_final':bool(FINAL_RE.search(str(text))), 'answer_kind':origin,
            'repetition':bool(re.search(r'(\b\w+\b)(?:\s+\1){2,}',str(text),re.I))}

def feat_bucket(f):return f['kind']+'|'+('final' if f['has_final'] else 'no_final')

# Essential semantic-failure regression checks (no GPU or network needed):
q={'kind':'arithmetic','target':14,'numbers':[]}
assert grade_output(r'We have \boxed{14}.',q)['success']
assert not grade_output(r'We have \boxed{14}.',q)['eligible_semantic_failure']
assert grade_output('FINAL_ANSWER: 13',q)['eligible_semantic_failure']
assert not grade_output('Then 14 times',q)['eligible_semantic_failure']
assert not grade_output('No answer yet',q)['eligible_semantic_failure']
assert not grade_output('FINAL_ANSWER: 13\n\\boxed{14}',q)['eligible_semantic_failure']
assert grade_output('The answer is 14',q)['success']
p={'kind':'countdown','numbers':[2,3,4],'target':14}
assert grade_output('2 + 3 * 4 = 14',p)['success']
assert not grade_output('2 + 3 * 4 = 14',p)['eligible_semantic_failure']
assert grade_output('2 + 3 + 4 = 9',p)['eligible_semantic_failure']
assert not grade_output('14',p)['eligible_semantic_failure']
assert not grade_output('2 + 4 + 4 = 14',p)['success']
print('Semantic grader regression suite PASSED: correct boxed excluded, explicit wrong admitted, truncations rejected.')

In [ ]:
sys.path.insert(0,REPO_DIR)
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_DIR=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
gc.collect();torch.cuda.empty_cache()
args=SimpleNamespace(model=MODEL_DIR,model_config=None,device='cuda',precision='fp32')
decoder=load_decoder(args);model=decoder.model;tokenizer=decoder.tokenizer
assert next(model.parameters()).dtype==torch.float32
print('Pinned ALoDLM loaded in FP32')

In [ ]:
def find_character_cut(ids,char_pos):
    n=len(ids)
    for j in range(1,n):
        if len(render(ids[:j]))>=char_pos:return max(1,j-1)
    return max(1,n//2)
def repair_cut(ids,action,source):
    n=len(ids)
    if action=='global':return 0
    if action.startswith('tail'):
        frac={'tail25':.75,'tail50':.50,'tail75':.25}[action]
        return max(1,min(n-1,int(n*frac)))
    text=render(ids)
    if action=='answer_locus':
        m=re.search(r'(?i)final_answer|(?<![A-Za-z])answer\s*(?:is|:)|\\boxed\s*\{',text)
        return max(1,min(n-1,find_character_cut(ids,m.start()) if m else n//2))
    if action=='first_math':
        m=re.search(r'(?:\d+\s*[+*/-]\s*\d+)|(?:\d+\s*=)',text)
        return max(1,min(n-1,find_character_cut(ids,m.start()) if m else n//2))
    raise ValueError(action)
assert repair_cut(list(range(40)),'tail25',{})==30
assert len({c['problem']['problem_id'] for c in CASES})==6
print('Boundary heuristics initialized; never use gold label to choose boundary.')

In [ ]:
SESSION_CAP=8;SESSION_MINS=27
started=time.monotonic();n_new=0;errors=[]
for item in PLAN:
    if n_new>=SESSION_CAP or time.monotonic()-started>=SESSION_MINS*60:break
    pid=item['problem_id'];action=item['action'];mode=item['decode']
    tid=f'E08|{pid}|{action}|{mode}|62083'
    if checked_load(tid) is not None:continue
    case=CASE_BY_ID[pid];p=case['problem'];src=case['source']
    ids=list(map(int,src['token_ids']));assert 8<=len(ids)<=192
    # H generation was conditioned on p.prompt + H mode-specific suffix.
    prompt=p['prompt']+PROMPTS[src['mode']]
    cut=repair_cut(ids,action,src);prefix=ids[:cut];budget=len(ids)-cut
    try:
        seed_all(62083)
        cfg=DecodeConfig(q=Q,tau=TAU,mode=mode,window_size=WINDOW,max_new_tokens=budget,position_penalty=None);cfg.validate()
        torch.cuda.reset_peak_memory_stats();t0=time.monotonic()
        out=decoder.generate(make_prompt(prompt)+prefix,cfg)
        new_ids=list(map(int,out['token_ids']));repaired=render(prefix+new_ids)
        ev=grade_output(repaired,p)
        rec={'trial_id':tid,'protocol_hash':PROTOCOL_HASH,'source_trial_id':src['trial_id'],
          'problem_id':pid,'kind':p['kind'],'split':'calibration','action':action,'decode':mode,
          'preserved_tokens':cut,'input_tokens':len(ids),'requested_new_tokens':budget,
          'generated_tokens':int(out.get('generated_tokens',len(new_ids))),'loop_token_passes':int(out.get('loop_token_passes',0)),
          'outer_steps':int(out.get('outer_steps',0)),'stop_reason':str(out.get('stop_reason','unknown')),
          'wall_seconds':round(time.monotonic()-t0,3),'peak_vram_gb':float(torch.cuda.max_memory_allocated()/2**30),
          'success':bool(ev['success']),'evaluation':ev,'repaired_output':repaired,
          'exact_prefix_invariant':(prefix+new_ids)[:cut]==prefix}
        assert rec['exact_prefix_invariant'];verify_save(rec);n_new+=1
        print('NEW',n_new,mode,action,pid,'success',ev['success'],flush=True)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache();errors.append({'trial_id':tid,'error':'OOM'});break
    except Exception as e:
        errors.append({'trial_id':tid,'error':repr(e)});break
if errors:atomic_json(ROOT/'latest_errors.json',errors)
print('New trials:',n_new,'Total:',len(recorded()),'/',len(PLAN))

In [ ]:
rows=[]
for r in recorded():
 rows.append({'problem_id':r['problem_id'],'decode':r['decode'],'action':r['action'],'success':int(r['success']),
   'preserved_tokens':r['preserved_tokens'],'forward_passes':r['loop_token_passes'],'wall_seconds':r['wall_seconds']})
if rows:
 df=pd.DataFrame(rows);display(df.groupby(['decode','action']).agg(trials=('success','size'),successes=('success','sum'),
  unique_problems=('problem_id','nunique'),mean_forward_passes=('forward_passes','mean')).reset_index())
 df.to_csv(ROOT/'raw_results.csv',index=False)
 summary=df.groupby(['decode','action']).agg(trials=('success','size'),successes=('success','sum'),
  unique_problems=('problem_id','nunique'),mean_forward_passes=('forward_passes','mean')).reset_index()
 summary.to_csv(ROOT/'policy_summary.csv',index=False)
atomic_json(ROOT/'progress.json',{'planned':len(PLAN),'completed':len(recorded()),'exploratory':True,
 'unique_calibration_problems':len(CASES),'missing':[f"E08|{x['problem_id']}|{x['action']}|{x['decode']}|62083" for x in PLAN if checked_load(f"E08|{x['problem_id']}|{x['action']}|{x['decode']}|62083") is None]})
snapshot_export('RSX_E08_RepairLocality_Checkpoint.zip')